In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 物流的发货清单
# 财务的发货清单
# 核算价
# PLM的生命周期全表
### 输出的所有文件
# 统计周期内产品核算价汇总 用于物料精简报告，因为里面有国内国外
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾  ，用于低效-长尾报告，里面只有国内
# 单型号贡献-统计值



### MAP关系汇总1、渠道对照 2、最终表格产品类别对应的产品组集合 3、产品组集合

In [2]:
# 物流的渠道对照关系清洗用
month = 202511
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'无',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'无',
'每誉':'每誉',
'渠道':'无',
'海外':'海外',
'调出渠道':'无',
'非零售工程电商':'非零售工程电商',
'无':'无'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 开票维度

In [23]:
df_income = pd.read_excel(r'C:\Users\zhangbon\Desktop\2024 2025收入.xlsx')
df_product = pd.read_excel(r"D:\000物料报表\202511\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_product['物料号'] = df_product['物料号'].astype(str).str[:13]
df_income['物料编码'] = df_income['物料编码'].astype(str).str[:13]
df_income['年月'] = pd.to_datetime(df_income['年月']).dt.strftime('%Y-%m-01')

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [44]:
# 处理一下income里面的气源，把物料编码进行合并
df_merge_gas = pd.read_excel(r'C:\Users\zhangbon\Desktop\气源地区对应关系20251105.XLSX',sheet_name='开票')
df_merge_gas['物料编码'] = df_merge_gas['物料编码'].astype(str).str[:13]
df_merge_gas_map_data = df_merge_gas.groupby('新标准型号',as_index=False).agg(
    物料编码 = ('物料编码','max'),
    新标准型号下的所有物料编码 = ('物料编码','unique')
)
merge_gas_map = {}
for index,row in df_merge_gas_map_data.iterrows():
    for item in row['新标准型号下的所有物料编码']:
        merge_gas_map[item] = row['物料编码']
df_merge_gas_map_data
# merge_gas_map
df_income['物料编码'] = df_income['物料编码'].apply(lambda x: merge_gas_map.get(x,x))

In [45]:
#只要2024年11月到2025年10月的数据、渠道筛选、产品线筛选、核算价不为0的
df_calu = df_income[(df_income['年月'] >= '2024-11-01') & 
                    (df_income['年月'] <= '2025-10-01') & 
                    (df_income['业务线'].isin(['零售','工程','电商'])) &
                    (df_income['产品线'].isin(['油烟机产品线','烹饪厨电产品线','洗碗机产品线','冰储产品线','净热产品线',])) & 
                    (df_income['核算价金额总计']!=0)].reset_index(drop=True)
df_calu.head()


,年,年月,产品线,任务分类,产品类别,物料编码,物料名称,业务线,数量,不含税收入总计,...,产品类别（手工分类）,系列1,系列2,N代,项目1,项目2,项目上市时间,项目ADCP时间,套系,山头项目
0,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100000,BCD-502WJLPAT,零售,174.0,1.290168e+06,...,家用冰箱,十字,502升,一代,P-2021057-RAAA2001,P-2021057-RAAA2001,2022-04-15,2022-08-31,非套系,0
1,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100000,BCD-502WJLPAT,工程,-1.0,-5.792040e+03,...,家用冰箱,十字,502升,一代,P-2021057-RAAA2001,P-2021057-RAAA2001,2022-04-15,2022-08-31,非套系,0
2,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100000,BCD-502WJLPAT,电商,1.0,7.322871e+03,...,家用冰箱,十字,502升,一代,P-2021057-RAAA2001,P-2021057-RAAA2001,2022-04-15,2022-08-31,非套系,0
3,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100001,BCD-502WYHPAT,零售,194.0,1.459579e+06,...,家用冰箱,十字,502升,一代,P-2024079-一代十字玥影灰项目,P-2024079-一代十字玥影灰项目,2024-11-26,2024-11-26,非套系,0
4,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100001,BCD-502WYHPAT,工程,5.0,2.540257e+04,...,家用冰箱,十字,502升,一代,P-2024079-一代十字玥影灰项目,P-2024079-一代十字玥影灰项目,2024-11-26,2024-11-26,非套系,0


In [46]:
# 产品维度——聚合算出每个产品的总销售量和金额
df_group1 = df_calu.groupby('物料编码',as_index=False).agg(
                                                        产品线=('产品线','first'),
                                                        产品总销售数量=('数量','sum'),
                                                        产品总收入=('核算价金额总计','sum'))
df_group1

,物料编码,产品线,产品总销售数量,产品总收入
0,1001000100114,油烟机产品线,-1.0,-1900.0
1,1001000200030,油烟机产品线,-13.0,-153400.0
2,1001000200040,油烟机产品线,-1.0,-9000.0
3,1001000200087,油烟机产品线,16.0,310400.0
4,1001000300046,油烟机产品线,-3.0,-6114.0
...,...,...,...,...
1503,1124000202970,洗碗机产品线,6.0,23400.0
1504,1124000202990,洗碗机产品线,8.0,12000.0
1505,1124000203000,洗碗机产品线,8.0,29600.0
1506,1124000204850,洗碗机产品线,1.0,2980.0


In [47]:
# 依据算出的依据物料编码聚合的产品总销售量和金额表，来计算累计销售金额占比、用于找出哪些型号贡献最大
df_out1 = df_group1.sort_values(by='产品总收入', ascending=False).reset_index(drop=True)
df_out1['产品累计收入'] = df_out1['产品总收入'].cumsum()
df_out1['所有产品总收入'] = df_out1['产品总收入'].sum()
df_out1['产品累计收入占比'] = df_out1['产品累计收入']/ df_out1['所有产品总收入']
df_out1['是否符合2080法则'] = df_out1['产品累计收入占比'] >= 0.8
df_out1


,物料编码,产品线,产品总销售数量,产品总收入,产品累计收入,所有产品总收入,产品累计收入占比,是否符合2080法则
0,1001001500097,油烟机产品线,232850.0,612321372.0,6.123214e+08,1.661581e+10,0.036852,False
1,1001000900395,油烟机产品线,153648.0,382276224.0,9.945976e+08,1.661581e+10,0.059858,False
2,1001002000029,油烟机产品线,100561.0,315764388.0,1.310362e+09,1.661581e+10,0.078862,False
3,1001001500106,油烟机产品线,101710.0,304309872.0,1.614672e+09,1.661581e+10,0.097177,False
4,1002003700039,烹饪厨电产品线,161255.0,279201240.0,1.893873e+09,1.661581e+10,0.113980,False
...,...,...,...,...,...,...,...,...
1503,1001001500032,油烟机产品线,-145.0,-718910.0,1.662054e+10,1.661581e+10,1.000284,True
1504,1005000400002,烹饪厨电产品线,-141.0,-722010.0,1.661982e+10,1.661581e+10,1.000241,True
1505,1005000400020,烹饪厨电产品线,-147.0,-740880.0,1.661908e+10,1.661581e+10,1.000196,True
1506,1019000200003,冰储产品线,-170.0,-1455540.0,1.661762e+10,1.661581e+10,1.000109,True


In [48]:
# df_result1_temp是只留下了贡献了80%收入的型号
product_line = {'油烟机产品线':0,'烹饪厨电产品线':1,'洗碗机产品线':2,'冰储产品线':3,'净热产品线':4,}
i_index1 = df_out1[df_out1['是否符合2080法则']==False].index.max() + 1
df_result1_temp = df_out1.loc[:i_index1]
df_result1_temp

,物料编码,产品线,产品总销售数量,产品总收入,产品累计收入,所有产品总收入,产品累计收入占比,是否符合2080法则
0,1001001500097,油烟机产品线,232850.0,612321372.0,6.123214e+08,1.661581e+10,0.036852,False
1,1001000900395,油烟机产品线,153648.0,382276224.0,9.945976e+08,1.661581e+10,0.059858,False
2,1001002000029,油烟机产品线,100561.0,315764388.0,1.310362e+09,1.661581e+10,0.078862,False
3,1001001500106,油烟机产品线,101710.0,304309872.0,1.614672e+09,1.661581e+10,0.097177,False
4,1002003700039,烹饪厨电产品线,161255.0,279201240.0,1.893873e+09,1.661581e+10,0.113980,False
...,...,...,...,...,...,...,...,...
185,1001000500376,油烟机产品线,10299.0,19879030.0,1.322830e+10,1.661581e+10,0.796127,False
186,1002003500064,烹饪厨电产品线,8542.0,19849920.0,1.324815e+10,1.661581e+10,0.797322,False
187,1001003300001,油烟机产品线,8063.0,19818854.0,1.326797e+10,1.661581e+10,0.798515,False
188,1008000800030,洗碗机产品线,3281.0,19686000.0,1.328765e+10,1.661581e+10,0.799699,False


In [49]:
# 依据贡献高的型号表，再看每个产品线各贡献了多少个型号，以及
df_result1 = df_result1_temp.groupby('产品线',as_index=False).agg(                                    
                                                                各产品线主力型号数 = ('物料编码','nunique') ,
                                                                各产品线主力型号收入 = ('产品总收入','sum')
).sort_values('产品线',key=lambda x:x.map(product_line)).reset_index(drop=True)
df_result1['主力型号总数'] = df_result1['各产品线主力型号数'].sum()
df_result1['主力型号总收入'] = df_result1['各产品线主力型号收入'].sum()
df_result1['各产品线主力型号数占比'] = df_result1['各产品线主力型号数'] / df_result1['主力型号总数']
df_result1['各产品线主力型号收入占比'] = df_result1['各产品线主力型号收入'] / df_result1['主力型号总收入']
df_result1['产品型号总数'] = df_group1['物料编码'].nunique()
df_result1['所有产品型号总收入'] = df_group1['产品总收入'].sum()
df_result1['主力型号数占比'] = df_result1['主力型号总数'] / df_result1['产品型号总数']
df_result1['主力型号收入占比'] = df_result1['主力型号总收入'] / df_result1['所有产品型号总收入']
df_result1

,产品线,各产品线主力型号数,各产品线主力型号收入,主力型号总数,主力型号总收入,各产品线主力型号数占比,各产品线主力型号收入占比,产品型号总数,所有产品型号总收入,主力型号数占比,主力型号收入占比
0,油烟机产品线,74,6.651486e+09,190,1.330708e+10,0.389474,0.499846,1508,1.661581e+10,0.125995,0.800869
1,烹饪厨电产品线,62,4.337285e+09,190,1.330708e+10,0.326316,0.325938,1508,1.661581e+10,0.125995,0.800869
2,洗碗机产品线,30,1.573366e+09,190,1.330708e+10,0.157895,0.118235,1508,1.661581e+10,0.125995,0.800869
3,冰储产品线,16,4.605730e+08,190,1.330708e+10,0.084211,0.034611,1508,1.661581e+10,0.125995,0.800869
4,净热产品线,8,2.843740e+08,190,1.330708e+10,0.042105,0.021370,1508,1.661581e+10,0.125995,0.800869


In [50]:
# 依据主力型号表，来计算都有哪些生命周期，贡献是多少
df_result1_temp['物料编码'] = df_result1_temp['物料编码'].astype(str).str[:13]
df_result1_temp['生命周期状态'] = df_result1_temp['物料编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))
df_result3 = df_result1_temp.groupby('生命周期状态',as_index=False).agg(
                                                        各状态主力型号数 = ('物料编码','nunique'),
                                                        各状态主力型号收入 = ('产品总收入','sum')                                                      
)
df_result3['主力型号总数'] = df_result3['各状态主力型号数'].sum()
df_result3['主力型号总收入'] = df_result3['各状态主力型号收入'].sum()
df_result3['各状态主力型号数占比'] = df_result3['各状态主力型号数']/df_result3['主力型号总数']
df_result3['各状态主力型号收入占比'] = df_result3['各状态主力型号收入']/df_result3['主力型号总收入']
df_result3

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_12196\3488797145.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result1_temp['物料编码'] = df_result1_temp['物料编码'].astype(str).str[:13]
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_12196\3488797145.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result1_temp['生命周期状态'] = df_result1_temp['物料编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))


,生命周期状态,各状态主力型号数,各状态主力型号收入,主力型号总数,主力型号总收入,各状态主力型号数占比,各状态主力型号收入占比
0,停止销售,18,7.837363e+08,190,1.330708e+10,0.094737,0.058896
1,退市预警,28,3.815860e+09,190,1.330708e+10,0.147368,0.286754
2,量产,144,8.707488e+09,190,1.330708e+10,0.757895,0.654350


In [51]:
# 按照产品线-物料编码维度，计算累计销售金额，用于每个产品线内部判断是否是主力型号
df_out2 = df_group1.sort_values(by=['产品线','产品总收入'], ascending=False).reset_index(drop=True)
df_out2['各产品线产品累计收入'] = df_out2.groupby('产品线')['产品总收入'].transform('cumsum')
df_out2['各产品线所有产品总收入'] = df_out2.groupby('产品线')['产品总收入'].transform('sum')
df_out2['各产品线产品累计收入占比'] = df_out2['各产品线产品累计收入'] / df_out2['各产品线所有产品总收入']
df_out2['是否符合2080法则'] = df_out2['各产品线产品累计收入占比'] >= 0.8
df_out2


,物料编码,产品线,产品总销售数量,产品总收入,各产品线产品累计收入,各产品线所有产品总收入,各产品线产品累计收入占比,是否符合2080法则
0,1002003700039,烹饪厨电产品线,161255.0,279201240.0,2.792012e+08,5.447020e+09,0.051258,False
1,1002003700037,烹饪厨电产品线,154239.0,267027230.0,5.462285e+08,5.447020e+09,0.100280,False
2,1002004300034,烹饪厨电产品线,148584.0,227333520.0,7.735620e+08,5.447020e+09,0.142016,False
3,1002003400077,烹饪厨电产品线,232366.0,186813330.0,9.603753e+08,5.447020e+09,0.176312,False
4,1002004300026,烹饪厨电产品线,139813.0,162274720.0,1.122650e+09,5.447020e+09,0.206104,False
...,...,...,...,...,...,...,...,...
1503,1003000200010,冰储产品线,-49.0,-89670.0,6.510639e+08,6.473218e+08,1.005781,True
1504,1003000700001,冰储产品线,-64.0,-209920.0,6.508540e+08,6.473218e+08,1.005457,True
1505,1003000600006,冰储产品线,-73.0,-270100.0,6.505839e+08,6.473218e+08,1.005039,True
1506,1019000200003,冰储产品线,-170.0,-1455540.0,6.491283e+08,6.473218e+08,1.002791,True


In [52]:
# 按照产品线维度，计算产品线收入、产品线总型号数、主力型号数
df_result2 = df_out2.groupby('产品线',as_index=False).agg(
                                        各产品线主力型号数 = ('是否符合2080法则',lambda x: (x==False).sum() + 1),
                                        各产品线主力型号收入 = ('产品总收入',lambda x: x[df_out2.loc[x.index,'是否符合2080法则']==False].sum() + (x[df_out2.loc[x.index, '是否符合2080法则'] == True].iloc[0])),
                                        各产品线总型号数 = ('物料编码','nunique'),
                                        各产品线总收入 = ('产品总收入','sum'),                                       
).sort_values(by='产品线',key=lambda x: x.map(product_line)).reset_index(drop=True)
df_result2['各产品线主力型号数占比'] = df_result2['各产品线主力型号数'] / df_result2['各产品线总型号数']
df_result2['各产品线主力型号收入占比'] = df_result2['各产品线主力型号收入'] / df_result2['各产品线总收入']
df_result2

,产品线,各产品线主力型号数,各产品线主力型号收入,各产品线总型号数,各产品线总收入,各产品线主力型号数占比,各产品线主力型号收入占比
0,油烟机产品线,54,6.173363e+09,451,7.682855e+09,0.119734,0.803525
1,烹饪厨电产品线,64,4.375262e+09,478,5.447020e+09,0.133891,0.803239
2,洗碗机产品线,38,1.710457e+09,250,2.134049e+09,0.152000,0.801508
3,冰储产品线,20,5.211250e+08,101,6.473218e+08,0.198020,0.805048
4,净热产品线,31,5.694880e+08,228,7.045674e+08,0.135965,0.808280


In [53]:
with pd.ExcelWriter(r'C:\Users\zhangbon\Desktop\单型号贡献_开票维度_合并气源.xlsx') as writer:
    df_out1.to_excel(writer, sheet_name='所有产品维度', index=False)
    df_out2.to_excel(writer, sheet_name='产品线——产品维度', index=False)
    df_result1.to_excel(writer, sheet_name='所有产品维度_主力型号', index=False)
    df_result2.to_excel(writer, sheet_name='产品线——产品维度_主力型号', index=False)
    df_result3.to_excel(writer, sheet_name='生命周期状态维度_主力型号', index=False)
    

### 物流维度

In [3]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2025年10月明细.xlsx', '10月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年10月明细.xlsx'), ('2025年1月明细.xlsx', '1月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年1月明细.xlsx'), ('2025年2月明细.xlsx', '2月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年2月明细.xlsx'), ('2025年3月明细.xlsx', '3月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年3月明细.xlsx'), ('2025年4月明细.xlsx', '4月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年4月明细.xlsx'), ('2025年5月明细.xlsx', '5月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年5月明细.xlsx'), ('2025年6月明细.xlsx', '6月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年6月明细.xlsx'), ('2025年7月明细.xlsx', '7月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年7月明细.xlsx'), ('2025年8月明细.xlsx', '8月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年8月明细.xlsx'), ('2025年9月明细.xlsx', '9月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年9月明细.xlsx')]


In [5]:
#如果有报错请提示报错信息
df = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量']]
    df = pd.concat([df, df_temp], axis=0).reset_index(drop=True)
df = df.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df['商品编码'] = df['商品编码'].astype(str)
df['渠道'].value_counts()

成功读取2025年10月明细.xlsx的10月明细表
成功读取2025年1月明细.xlsx的1月明细表
成功读取2025年2月明细.xlsx的2月明细表
成功读取2025年3月明细.xlsx的3月明细表
成功读取2025年4月明细.xlsx的4月明细表
成功读取2025年5月明细.xlsx的5月明细表
成功读取2025年6月明细.xlsx的6月明细表
成功读取2025年7月明细.xlsx的7月明细表
成功读取2025年8月明细.xlsx的8月明细表
成功读取2025年9月明细.xlsx的9月明细表


渠道
零售        390588
工程         16940
电商          7855
电商不可售       6616
海外          1771
每誉           152
战略电商         129
新品            67
内部处理通用        15
借出渠道          14
商净             7
渠道             4
调出渠道           2
转出渠道           2
米博新零售          1
Name: count, dtype: int64

In [6]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\财务发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2024年11月明细.xlsx', '11月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\财务发货\\2024年11月明细.xlsx'), ('2024年12月明细.xlsx', '12月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\财务发货\\2024年12月明细.xlsx')]


In [7]:
caiwu_shouru = {'商品编码':[],'渠道':[],'实际出库数量':[]}
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(file[0], file[1])
    for index,row in df_temp.iterrows():
        if row['零售'] > 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('零售')
            caiwu_shouru['实际出库数量'].append(row['零售'])
        if row['工程'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('工程')
            caiwu_shouru['实际出库数量'].append(row['工程'])
        if row['电商'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('电商')
            caiwu_shouru['实际出库数量'].append(row['电商'])
        if row['合计-发货'] - row['零售'] - row['工程'] - row['电商'] != 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('非零售工程电商')
            caiwu_shouru['实际出库数量'].append(row['合计-发货'] - row['零售'] - row['工程'] - row['电商'])

df_caiwu = pd.DataFrame(caiwu_shouru)
df_caiwu['商品编码'] = df_caiwu['商品编码'].astype(str)
df_caiwu['渠道'].value_counts()

成功读取2024年11月明细.xlsx的11月明细表
成功读取2024年12月明细.xlsx的12月明细表


渠道
零售         1029
电商          797
工程          476
非零售工程电商      16
Name: count, dtype: int64

In [28]:
df0 = pd.concat([df, df_caiwu], axis=0).reset_index(drop=True)
df0['商品编码'] = df0['商品编码'].astype(str).str[:13]
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[df0['渠道'].isin(['零售','工程','电商'])].reset_index(drop=True)
df0.info()
# df0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 424512 entries, 0 to 424511
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   商品编码    424512 non-null  object
 1   渠道      424512 non-null  object
 2   实际出库数量  424512 non-null  object
dtypes: object(3)
memory usage: 9.7+ MB


In [29]:
df1 = df0.copy()
df1[['商品编码','渠道']] = df1[['商品编码','渠道']].astype(str)
df1['实际出库数量'] = df1['实际出库数量'].astype(float)

# PLM产品数据导入
df_product_group = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_product_group[['物料号','标准型号','国内/海外','产品线']] = df_product_group[['物料号','标准型号','国内/海外','产品线']].astype(str)
df_product_group = df_product_group[df_product_group['物料号'].str.len()>10]
df_product_group['物料号'] = df_product_group['物料号'].apply(lambda x: x[:13])
# df_product_group

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [30]:
#产品组对照关系
product_group_map = dict(zip(df_product_group['物料号'],df_product_group['产品组']))
#标准型号对照关系
product_standard = dict(zip(df_product_group['物料号'],df_product_group['标准型号']))
#记录国内/海外状态
product_country = dict(zip(df_product_group['物料号'],df_product_group['国内/海外']))
#记录产品线
product_line = dict(zip(df_product_group['物料号'],df_product_group['产品线'])) 


In [31]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['产品编码'] = df_price['产品编码'].astype(str)
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
product_price_map = dict(zip(df_price['产品编码'],df_price['系统核算价']))

In [32]:
df1['产品组'] = df1['商品编码'].map(product_group_map)
df1['产品线'] = df1['商品编码'].map(product_line)
df1['系统核算价'] = df1['商品编码'].map(product_price_map)
df1['核算价'] = df1['系统核算价'] * df1['实际出库数量']
df1['标准型号'] = df1['商品编码'].map(product_standard)
df1['国内/海外'] = df1['商品编码'].map(product_country)
df1 = df1[df1['商品编码'].str.startswith('10')].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'国内没有系统核算价的是这些数据\n{df1[(df1['系统核算价'].isnull()&(df1['国内/海外']=='国内'))]['商品编码'].drop_duplicates()}')
print(len(df1))
df1['核算价'] = df1['核算价']/10000
df1

国内没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
419684


,商品编码,渠道,实际出库数量,产品组,产品线,系统核算价,核算价,标准型号,国内/海外
0,1001001500116,零售,12.0,吸油烟机,油烟机产品线,3358.0,4.0296,Z8T,国内
1,1009000600033,零售,1.0,蒸烤烹饪机,烹饪厨电产品线,3450.0,0.3450,ZK50-02-F1,国内
2,1009000500035,零售,3.0,灶蒸烤烹饪机,烹饪厨电产品线,5280.0,1.5840,JZT-ZK46-X2,国内
3,1001001500131,零售,6.0,吸油烟机,油烟机产品线,2988.0,1.7928,02-Z6TA,国内
4,1002003700049,零售,5.0,灶具,烹饪厨电产品线,2550.0,1.2750,H8B,国内
...,...,...,...,...,...,...,...,...,...
419679,1019000300007,工程,2.0,家用冰箱,冰储产品线,26923.0,5.3846,BCD-508W-Y1Pro,国内
419680,1019000200004,零售,130.0,家用冰箱,冰储产品线,8562.0,111.3060,BCD-510WYYPAF,国内
419681,1019000200001,零售,15.0,家用冰箱,冰储产品线,8562.0,12.8430,BCD-510WZBPAF,国内
419682,1019000200001,电商,2.0,家用冰箱,冰储产品线,8562.0,1.7124,BCD-510WZBPAF,国内


In [53]:
qudao = ['工程']
df_cleaned = df1[(df1[r'国内/海外']=='国内')&
                (df1['渠道'].isin(qudao))&
                (df1['产品线'].isin(['油烟机产品线','烹饪厨电产品线','洗碗机产品线','冰储产品线','净热产品线',]))
    ].reset_index(drop=True)
df_cleaned

,商品编码,渠道,实际出库数量,产品组,产品线,系统核算价,核算价,标准型号,国内/海外
0,1018000900006,工程,1.0,嵌入式洗碗机,洗碗机产品线,2750.0,0.2750,JPCD12E-P3S,国内
1,1003000300039,工程,8.0,消毒柜,冰储产品线,1300.0,1.0400,ZTD110J-01-J53,国内
2,1001000800357,工程,112.0,吸油烟机,油烟机产品线,1588.0,17.7856,EH51,国内
3,1018000900006,工程,14.0,嵌入式洗碗机,洗碗机产品线,2750.0,3.8500,JPCD12E-P3S,国内
4,1001000800341,工程,146.0,吸油烟机,油烟机产品线,1588.0,23.1848,EH51,国内
...,...,...,...,...,...,...,...,...,...
16426,1019000300001,工程,3.0,家用冰箱,冰储产品线,14286.0,4.2858,BCD-508W-X20.i,国内
16427,1019000300003,工程,2.0,家用冰箱,冰储产品线,10990.0,2.1980,BCD-508W-S3-JH,国内
16428,1019000300004,工程,17.0,家用冰箱,冰储产品线,16667.0,28.3339,BCD-508W-S7-CM,国内
16429,1019000300005,工程,2.0,家用冰箱,冰储产品线,10990.0,2.1980,BCD-508W-S3-NB,国内


In [54]:
# 把发货维度的气源进行合并
df_merge_gas = pd.read_excel(r'C:\Users\zhangbon\Desktop\气源地区对应关系20251105.XLSX',sheet_name='发货')
df_merge_gas['物料编码'] = df_merge_gas['物料编码'].astype(str).str[:13]
df_merge_gas_map_data = df_merge_gas.groupby('新标准型号',as_index=False).agg(
    物料编码 = ('物料编码','max'),
    新标准型号下的所有物料编码 = ('物料编码','unique')
)
merge_gas_map = {}
for index,row in df_merge_gas_map_data.iterrows():
    for item in row['新标准型号下的所有物料编码']:
        merge_gas_map[item] = row['物料编码']
# merge_gas_map
df_cleaned['商品编码'] = df_cleaned['商品编码'].apply(lambda x: merge_gas_map.get(x,x))
df_cleaned

,商品编码,渠道,实际出库数量,产品组,产品线,系统核算价,核算价,标准型号,国内/海外
0,1018000900006,工程,1.0,嵌入式洗碗机,洗碗机产品线,2750.0,0.2750,JPCD12E-P3S,国内
1,1003000300039,工程,8.0,消毒柜,冰储产品线,1300.0,1.0400,ZTD110J-01-J53,国内
2,1001000800357,工程,112.0,吸油烟机,油烟机产品线,1588.0,17.7856,EH51,国内
3,1018000900006,工程,14.0,嵌入式洗碗机,洗碗机产品线,2750.0,3.8500,JPCD12E-P3S,国内
4,1001000800341,工程,146.0,吸油烟机,油烟机产品线,1588.0,23.1848,EH51,国内
...,...,...,...,...,...,...,...,...,...
16426,1019000300001,工程,3.0,家用冰箱,冰储产品线,14286.0,4.2858,BCD-508W-X20.i,国内
16427,1019000300003,工程,2.0,家用冰箱,冰储产品线,10990.0,2.1980,BCD-508W-S3-JH,国内
16428,1019000300004,工程,17.0,家用冰箱,冰储产品线,16667.0,28.3339,BCD-508W-S7-CM,国内
16429,1019000300005,工程,2.0,家用冰箱,冰储产品线,10990.0,2.1980,BCD-508W-S3-NB,国内


In [55]:
# 产品维度，计算出每个产品的销售总数量和总金额
df_group2 = df_cleaned.groupby('商品编码',as_index=False).agg(
                                                            产品线=('产品线','first'),
                                                            产品总销售数量=('实际出库数量','sum'),
                                                            产品总收入=('核算价','sum'),
)
df_group2

,商品编码,产品线,产品总销售数量,产品总收入
0,1001000300073,油烟机产品线,88.0,19.4304
1,1001000500073,油烟机产品线,108.0,30.8664
2,1001000500224,油烟机产品线,4938.0,1470.5364
3,1001000500251,油烟机产品线,476.0,125.0928
4,1001000500283,油烟机产品线,5387.0,1254.0936
...,...,...,...,...
374,1019000300005,冰储产品线,18.0,19.7820
375,1019000300006,冰储产品线,8.0,11.4288
376,1019000300007,冰储产品线,8.0,21.5384
377,1024000200018,洗碗机产品线,36.0,8.6400


In [56]:
# 对产品维度的表进行金额降序排序再进行计算累计求和金额，用于判断主力型号
df_out3 = df_group2.sort_values(by='产品总收入',ascending=False).reset_index(drop=True)
df_out3['产品累计收入'] = df_out3['产品总收入'].cumsum()
df_out3['所有产品总销售金额'] = df_out3['产品总收入'].sum()
df_out3['产品累计收入占比'] = df_out3['产品累计收入']/df_out3['所有产品总销售金额']
df_out3['是否符合2080法则'] = df_out3['产品累计收入占比'] >= 0.8
df_out3

,商品编码,产品线,产品总销售数量,产品总收入,产品累计收入,所有产品总销售金额,产品累计收入占比,是否符合2080法则
0,1018000900006,洗碗机产品线,37809.0,10397.4750,10397.4750,208107.1228,0.049962,False
1,1002003400104,烹饪厨电产品线,89743.0,7807.6410,18205.1160,208107.1228,0.087480,False
2,1002003400077,烹饪厨电产品线,92624.0,7317.2960,25522.4120,208107.1228,0.122641,False
3,1001000800353,油烟机产品线,46705.0,6949.7040,32472.1160,208107.1228,0.156036,False
4,1001000800341,油烟机产品线,37138.0,5897.5144,38369.6304,208107.1228,0.184374,False
...,...,...,...,...,...,...,...,...
374,1002000400052,烹饪厨电产品线,2.0,0.2620,208106.4943,208107.1228,0.999997,True
375,1001000500375,油烟机产品线,1.0,0.1930,208106.6873,208107.1228,0.999998,True
376,1004001300011,净热产品线,1.0,0.1775,208106.8648,208107.1228,0.999999,True
377,1001000900372,油烟机产品线,1.0,0.1500,208107.0148,208107.1228,0.999999,True


In [57]:
# 依据产品维度把主力型号给筛选出来
i_index3 = df_out3[df_out3['是否符合2080法则']==True].index.min()
df_result2_temp = df_out3.loc[:i_index3]
df_result2_temp


,商品编码,产品线,产品总销售数量,产品总收入,产品累计收入,所有产品总销售金额,产品累计收入占比,是否符合2080法则
0,1018000900006,洗碗机产品线,37809.0,10397.4750,10397.4750,208107.1228,0.049962,False
1,1002003400104,烹饪厨电产品线,89743.0,7807.6410,18205.1160,208107.1228,0.087480,False
2,1002003400077,烹饪厨电产品线,92624.0,7317.2960,25522.4120,208107.1228,0.122641,False
3,1001000800353,油烟机产品线,46705.0,6949.7040,32472.1160,208107.1228,0.156036,False
4,1001000800341,油烟机产品线,37138.0,5897.5144,38369.6304,208107.1228,0.184374,False
...,...,...,...,...,...,...,...,...
67,1008000500017,洗碗机产品线,2460.0,824.1000,163549.9134,208107.1228,0.785893,False
68,1002003400127,烹饪厨电产品线,9113.0,820.1700,164370.0834,208107.1228,0.789834,False
69,1002003400146,烹饪厨电产品线,7000.0,805.0000,165175.0834,208107.1228,0.793702,False
70,1008000500021,洗碗机产品线,2360.0,790.6000,165965.6834,208107.1228,0.797501,False


In [58]:
# 依据产品维度的主力型号表，把主力型号总数、以及在各个产品线的分布计算出来
product_line = {'油烟机产品线':0,'烹饪厨电产品线':1,'洗碗机产品线':2,'冰储产品线':3,'净热产品线':4,}
df_result4 = df_result2_temp.groupby('产品线',as_index=False).agg(
                                                                各产品线主力型号数=('商品编码','nunique'),
                                                                各产品线主力型号收入=('产品总收入','sum')
).sort_values('产品线',key=lambda x: x.map(product_line)).reset_index(drop=True)
df_result4['主力型号总数'] = df_result4['各产品线主力型号数'].sum()
df_result4['主力型号总收入'] = df_result4['各产品线主力型号收入'].sum()
df_result4['各产品线主力型号数占比'] = df_result4['各产品线主力型号数'] / df_result4['主力型号总数']
df_result4['各产品线主力型号收入占比'] = df_result4['各产品线主力型号收入'] / df_result4['主力型号总收入']
df_result4['产品型号总数'] = df_group2['商品编码'].nunique()
df_result4['所有产品型号总收入'] = df_group2['产品总收入'].sum()
df_result4['主力型号数占比'] = df_result4['主力型号总数'] / df_result4['产品型号总数']
df_result4['主力型号收入占比'] = df_result4['主力型号总收入'] / df_result4['所有产品型号总收入']
df_result4

,产品线,各产品线主力型号数,各产品线主力型号收入,主力型号总数,主力型号总收入,各产品线主力型号数占比,各产品线主力型号收入占比,产品型号总数,所有产品型号总收入,主力型号数占比,主力型号收入占比
0,油烟机产品线,33,69506.3134,72,166743.5834,0.458333,0.416846,379,208107.1228,0.189974,0.801239
1,烹饪厨电产品线,18,42206.4470,72,166743.5834,0.250000,0.253122,379,208107.1228,0.189974,0.801239
2,洗碗机产品线,15,44621.7330,72,166743.5834,0.208333,0.267607,379,208107.1228,0.189974,0.801239
3,冰储产品线,6,10409.0900,72,166743.5834,0.083333,0.062426,379,208107.1228,0.189974,0.801239


In [59]:
# 依据主力型号表，计算出生命周期状态分布以及每个状态的金额占比
df_result2_temp['商品编码'] = df_result2_temp['商品编码'].astype(str).str[:13]
df_result2_temp['生命周期状态'] = df_result2_temp['商品编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))
df_result6 = df_result2_temp.groupby('生命周期状态',as_index=False).agg(
                                                        各状态主力型号数 = ('商品编码','nunique'),
                                                        各状态主力型号收入 = ('产品总收入','sum')                                                      
)
df_result6['主力型号总数'] = df_result6['各状态主力型号数'].sum()
df_result6['主力型号总收入'] = df_result6['各状态主力型号收入'].sum()
df_result6['各状态主力型号数占比'] = df_result6['各状态主力型号数'] / df_result6['主力型号总数']
df_result6['各状态主力型号收入占比'] = df_result6['各状态主力型号收入']/df_result6['主力型号总收入']
df_result6


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_29320\2785767290.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result2_temp['商品编码'] = df_result2_temp['商品编码'].astype(str).str[:13]
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_29320\2785767290.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result2_temp['生命周期状态'] = df_result2_temp['商品编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))


,生命周期状态,各状态主力型号数,各状态主力型号收入,主力型号总数,主力型号总收入,各状态主力型号数占比,各状态主力型号收入占比
0,停止销售,18,39320.4820,72,166743.5834,0.250000,0.235814
1,退市预警,10,22747.1796,72,166743.5834,0.138889,0.136420
2,量产,44,104675.9218,72,166743.5834,0.611111,0.627766


In [60]:
# 按照产品线和产品总收入排序，然后计算各个产品线内的累计金额和占比，用于筛选每个产品线内的主力型号
df_out4 = df_group2.sort_values(by=['产品线','产品总收入'],ascending=False).reset_index(drop=True)
df_out4['产品累计收入'] = df_out4.groupby('产品线')['产品总收入'].cumsum()
df_out4['所有产品收入'] = df_out4.groupby('产品线')['产品总收入'].transform('sum')
df_out4['产品累计收入占比'] = df_out4['产品累计收入']/df_out4['所有产品收入']
df_out4['是否符合2080法则'] = df_out4['产品累计收入占比'] >= 0.8
df_out4


,商品编码,产品线,产品总销售数量,产品总收入,产品累计收入,所有产品收入,产品累计收入占比,是否符合2080法则
0,1002003400104,烹饪厨电产品线,89743.0,7807.641,7807.6410,56754.5139,0.137569,False
1,1002003400077,烹饪厨电产品线,92624.0,7317.296,15124.9370,56754.5139,0.266498,False
2,1002003400032,烹饪厨电产品线,55895.0,5030.550,20155.4870,56754.5139,0.355135,False
3,1009001100002,烹饪厨电产品线,8521.0,3024.955,23180.4420,56754.5139,0.408433,False
4,1002003400106,烹饪厨电产品线,32862.0,2300.340,25480.7820,56754.5139,0.448965,False
...,...,...,...,...,...,...,...,...
374,1003000600021,冰储产品线,3.0,0.669,13236.8829,13238.4039,0.999885,True
375,1003000400006,冰储产品线,2.0,0.490,13237.3729,13238.4039,0.999922,True
376,1003000400008,冰储产品线,1.0,0.355,13237.7279,13238.4039,0.999949,True
377,1003000900006,冰储产品线,1.0,0.342,13238.0699,13238.4039,0.999975,True


In [61]:
# 依据产品线-商品编码的金额汇总表，计算每个产品线的收入，以及产品型号的数量、还有主力型号的数量
df_result5 = df_out4.groupby('产品线',as_index=False).agg(
    各产品线主力型号数 = ('是否符合2080法则',lambda x:(x==False).sum() + 1),
    各产品线主力型号收入 = ('产品总收入',lambda x: x[df_out4.loc[x.index,'是否符合2080法则']==False].sum() + (x[df_out4.loc[x.index, '是否符合2080法则'] == True].iloc[0])),
    各产品线总型号数 = ('商品编码','nunique'),
    各产品线总收入 = ('产品总收入','sum'),
).sort_values(by='产品线',key = lambda x:x.map(product_line)).reset_index(drop=True)
df_result5['各产品线主力型号数占比'] = df_result5['各产品线主力型号数'] / df_result5['各产品线总型号数']
df_result5['各产品线主力型号收入占比'] = df_result5['各产品线主力型号收入'] / df_result5['各产品线总收入']
df_result5

,产品线,各产品线主力型号数,各产品线主力型号收入,各产品线总型号数,各产品线总收入,各产品线主力型号数占比,各产品线主力型号收入占比
0,油烟机产品线,34,70278.0142,111,87155.7408,0.306306,0.806350
1,烹饪厨电产品线,23,45634.5860,131,56754.5139,0.175573,0.804070
2,洗碗机产品线,10,39039.6880,62,48725.4500,0.161290,0.801218
3,冰储产品线,7,11160.5300,38,13238.4039,0.184211,0.843042
4,净热产品线,6,1790.1900,37,2233.0142,0.162162,0.801692


In [62]:
with pd.ExcelWriter(fr'C:\Users\zhangbon\Desktop\单型号贡献_发货维度_合并气源_{qudao}.xlsx') as writer:
    df_out3.to_excel(writer, sheet_name='产品维度', index=False)
    df_out4.to_excel(writer, sheet_name='产品线产品维度', index=False)
    df_result4.to_excel(writer, sheet_name='所有产品维度_主力型号', index=False)
    df_result5.to_excel(writer, sheet_name='产品线-产品维度_主力型号', index=False)
    df_result6.to_excel(writer, sheet_name='生命周期状态维度_主力型号', index=False)
